In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# # navigate to root directory of current file in order to access other files relatively
# %cd /content/drive/MyDrive/Colab\ Notebooks/thesis-writing-1/eda-signal-classifier

In [3]:
# !pip install PyWavelets

In [4]:
import datetime
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import requests

from utilities.loaders import charge_raw_data, download_dataset

from utilities.preprocessors import (
    which_element,
    find_begin_end,)

from utilities.visualizers import (
    view_time_frame,
    view_wavelet_coeffs,
    analyze,
    data_split_metric_values,
    view_value_frequency,
    multi_class_heatmap,
    view_metric_values,
    view_classified_labels,
    view_label_freq,
    disp_cat_feat,
    plot_all_features,
    describe_col,
    ModelResults,
    view_all_splits_results)

from utilities.feature_extractors import (
    concur_extract_features_from_all,
    extract_features_per_hour,
    load_wavelet_data,
    restructure_wavelets,
    compute_features,
    interpolate_signals,
    butter_lowpass_filter,
    get_time_frequency,
    rejoin_data,)

%load_ext autoreload
%autoreload 2

# Downloading dataset

If your project requires downloading a larger file, then you may run into issues using the steps above when you try to load the entire file into memory. To overcome those issues, you can download large files in a streaming fashion to avoid reading the content of large responses all at once

In [5]:
# download_dataset("https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/w8fxrg4pv5-2.zip")

# Loading dataset

In [6]:
# # Extract data from zip file
# with zipfile.ZipFile('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/EDABE dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)')

In [7]:
# ahixac_eda_df_128hz = pd.read_csv('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/Train/ahixac_expert1.csv', sep=';')
# ahixac_eda_df_128hz

In [8]:
# ahixac_eda_df_128hz.columns = ['time', 'raw_signal', 'clean_signal', 'label', 'auto_signal', 'pred_art', 'post_proc_pred_art']

In [9]:
# start_time = ahixac_eda_df_128hz.iloc[0]['time']
# start_time

In [10]:
# ahixac_eda_df_128hz.set_index(pd.date_range(start=start_time, periods=ahixac_eda_df_128hz.shape[0], freq=get_time_frequency(128)), inplace=True)
# ahixac_eda_df_128hz

# Downsampling 128hz signals to 16hz

In [11]:
# ahixac_eda_df_16hz = interpolate_signals(ahixac_eda_df_128hz, sample_rate=128, start_time=start_time, target_hz=16)
# ahixac_eda_df_16hz

# Low-pass filtering raw 128hz and 16hz signals

In [12]:
# ahixac_eda_df_128hz['filtered_signal'] = butter_lowpass_filter(ahixac_eda_df_128hz['raw_signal'], cutoff=1.0, samp_freq=128, order=6)
# ahixac_eda_df_16hz['filtered_signal'] = butter_lowpass_filter(ahixac_eda_df_16hz['raw_signal'], cutoff=1.0, samp_freq=16, order=6)

In [13]:
# ahixac_eda_df_128hz

In [14]:
# ahixac_eda_df_128hz.iloc[63]

In [15]:
# timestamp_list = ahixac_eda_df_128hz.index.tolist()[::64]
# timestamp_list

In [16]:
# timestamp_list[-1].timestamp()

In [17]:
# ahixac_eda_df_16hz

In [18]:
# ahixac_eda_df_16hz[:8]

In [19]:
# view_time_frame(ahixac_eda_df_128hz, samp_freq=128, cols_to_use=['raw_signal', 'filtered_signal'], img_title='subject ahixac 128hz time frame')
# view_time_frame(ahixac_eda_df_16hz, samp_freq=16, cols_to_use=['raw_signal', 'filtered_signal'], img_title='subject ahixac 16hz time frame')

# Iterate through signals per hour

In [20]:
# data_128hz = extract_features_per_hour(ahixac_eda_df_128hz, hertz=128, window_size=0.5, verbose=True)
# data_128hz

In [21]:
# data_16hz = extract_features_per_hour(ahixac_eda_df_16hz, hertz=16, window_size=0.5, verbose=True)
# data_16hz

#### if we had a 128hz dataset with derived timestamps that increase every 0.5s such as this [0.0, 0.5, 1.0, 1.5, ..., 6506.0] then our segments would be:
```
[0.0, 0.5)
[0.5, 1.0)
[1.0, 1.5)
...
[6504.5, 6506.0)
```

#### 832830 / 64 is 13012.96875 or when "`math.ceil()`ed" is 13013

In [22]:
# math.ceil(13012.96875), math.floor(13012.96875)

In [23]:
# for feature_segments, labels in data_128hz:
#     print(labels.value_counts())

#### here in the first hour of our data the number of artifacts out of all 7200 0.5s segments is 716 or roughly 9.9% of our data, and the number of non-artifacts out of all 7200 0.5s segments is 6484 or roughly 90% of our data

#### For the second hour of our data the number of artifacts out of all 5813 0.5s segments is 208 or roughly 3.58% of our data, and the number of non-artifacts out of all 5813 0.5s segments is 5605 or roughly 96.42% of our data

In [24]:
# for feature_segments, labels in data_16hz:
#     print(labels.value_counts())

#### Here the reason why we have almost the same number of artifact and non-artifact labels to the 128hz data is because we interpolated our 128hz data to 16hz thus losing some of our labels

In [25]:
# ahixac_eda_data = rejoin_data(data_128hz, data_16hz)
# ahixac_eda_data

#### concatenating calculated features from 128hz and 16hz data of the first hour

In [26]:
# ahixac_eda_data[0].columns

# Now we ought to do these for all subjects

# scanning train folder

In [27]:
train_files = os.listdir('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/Train/')
train_files

['ahixac_expert1.csv',
 'akakip_expert2.csv',
 'aqamom_expert2.csv',
 'aretez_expert1.csv',
 'asifex_expert2.csv',
 'axeyoh_expert2.csv',
 'efawep_expert2.csv',
 'egemow_expert2.csv',
 'ejofeq_expert2.csv',
 'erecij_expert1.csv',
 'esirur_expert1.csv',
 'ewehov_expert2.csv',
 'exozef_expert2.csv',
 'idagah_expert2.csv',
 'ihikay_expert1.csv',
 'ihinot_expert1.csv',
 'imocac_expert2.csv',
 'iqiyat_expert2.csv',
 'obujoh_expert2.csv',
 'ohayeh_expert1.csv',
 'ohufow_expert1.csv',
 'ojotew_expert1.csv',
 'onivuk_expert1.csv',
 'opunad_expert1.csv',
 'otecab_expert2.csv',
 'otuqom_expert1.csv',
 'owegud_expert2.csv',
 'oxisux_expert1.csv',
 'tchgij_expert2.csv',
 'ufoyek_expert2.csv',
 'uqozew_expert1.csv',
 'urogif_expert1.csv',
 'uzefow_expert1.csv']

# Concurrently read each .csv file and use functions that will spit out the features

In [28]:
train_eda_data = concur_extract_features_from_all('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/Train/', train_files)
train_eda_data

c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:857: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eda_df_16hz['filtered_signal'] = butter_lowpass_filter(eda_df_16hz['raw_signal'], cutoff=1.0, samp_freq=16, order=6)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:424: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  whole_inc_ts = pd.date_range(start=timestamp_list[0], periods=data.shape[0], freq=f'{((1 / whole_freq) * 1000)}ms')
c:\Users\LARRY\Documents\

processing hour 0 - start: 0 | end: 460800
processing hour 0 - start: 0 | end: 460800
processing hour 0 - start: 0 | end: 460800
processing hour 0 - start: 0 | end: 460800
processing hour 0 - start: 0 | end: 460800
processing hour 0 - start: 0 | end: 460800


c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:857: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eda_df_16hz['filtered_signal'] = butter_lowpass_filter(eda_df_16hz['raw_signal'], cutoff=1.0, samp_freq=16, order=6)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:424: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  whole_inc_ts = pd.date_range(start=timestamp_list[0], periods=data.shape[0], freq=f'{((1 / whole_freq) * 1000)}ms')
c:\Users\LARRY\Documents\

processing hour 0 - start: 0 | end: 460800


c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:857: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eda_df_16hz['filtered_signal'] = butter_lowpass_filter(eda_df_16hz['raw_signal'], cutoff=1.0, samp_freq=16, order=6)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:424: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  whole_inc_ts = pd.date_range(start=timestamp_list[0], periods=data.shape[0], freq=f'{((1 / whole_freq) * 1000)}ms')
c:\Users\LARRY\Documents\

...

c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:425: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  half_inc_ts = pd.date_range(start=timestamp_list[0], periods=data.shape[0], freq=f'{((1 / half_freq) * 1000)}ms')
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide

processing hour 0 - start: 0 | end: 460800


c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase

processing hour 0 - start: 0 | end: 460800


c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase = np.arctan(z_imag / z_real)
c:\Users\LARRY\Documents\Scripts\thesis-writing-1\eda-signal-classifier\utilities\feature_extractors.py:347: RuntimeWarning: divide by zero encountered in scalar divide
  phase

#### Above code takes about 204 minutes or 3 hrs and 20 minutes to run

In [ ]:
# save each feature dataframe as a .csv file in the folder created earlier with the same names
for subject_name, (feature_segments, labels)  in train_eda_data:
    feature_segments.to_csv(f'./data/Artifact Detection Data/train/{subject_name}_features.csv')
    labels.to_csv(f'./data/Artifact Detection Data/train/{subject_name}_labels.csv')

In [ ]:
test_files = os.listdir('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/Test/')
test_files

In [ ]:
test_eda_data = concur_extract_features_from_all('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)/Test/', test_files)
test_eda_data

In [ ]:
# save each feature dataframe as a .csv file in the folder created earlier with the same names
for subject_name, (feature_segments, labels)  in test_eda_data:
    feature_segments.to_csv(f'./data/Artifact Detection Data/test/{subject_name}_features.csv')
    labels.to_csv(f'./data/Artifact Detection Data/test/{subject_name}_labels.csv')

In [ ]:
# # once notebook reaches end remove data to clear space
# os.remove('./data/EDABE dataset.zip')
# os.remove('./data/Electrodermal Activity artifact correction BEnchmark (EDABE)')